In [ ]:
import argparse
import os
import random

import numpy as np
import torch
import torchvision.models as models
from torchvision import transforms

from attack_utils import run_experiment
from impl_cw import CWLinf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

datasets = {
    'correct_1000_AlexNet': ('alexnet', models.AlexNet_Weights.IMAGENET1K_V1),
    'correct_1000_DenseNet121': ('densenet121', models.DenseNet121_Weights.IMAGENET1K_V1),
    'correct_1000_GoogLeNet': ('googlenet', models.GoogLeNet_Weights.IMAGENET1K_V1),
    'correct_1000_MobileNetV3_large': ('mobilenet_v3_large', models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    'correct_1000_ResNet34': ('resnet34', models.ResNet34_Weights.IMAGENET1K_V1),
    'correct_1000_VGG11': ('vgg11', models.VGG11_Weights.IMAGENET1K_V1),
    'correct_1000_EfficientNet_b0': ('efficientnet_b0', models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}

config = {
    'SEED': 42,
    'selected_count': 500,
    'output_dir': 'adversarial_samples',
    'attack_name': 'cw',
    'MAX_SAVE_ADV': 10,
    'if_save_adv': False,
    'threshold': 1e-6,
    'target_labels': torch.tensor([100]).to(device),
    'if_target': False,
    'if_prune': False,
    # 攻击参数
    'eps': 100/255,
    'alpha': 1/255,
    'steps': 100,
    'random_start': False,
    'targeted': False,
    'kappa': 0.0,
    # 剪枝参数
    'step_ratio': 0.01,
    'max_ratio': 1.0,
}

parser = argparse.ArgumentParser(description='Adversarial Attack Experiment')
parser.add_argument('--eps', type=float, default=config['eps'], help='Epsilon for CW Linf')
parser.add_argument('--alpha', type=float, default=config['alpha'], help='Alpha for CW Linf')
parser.add_argument('--steps', type=int, default=config['steps'], help='Steps for CW Linf')
parser.add_argument('--kappa', type=float, default=config['kappa'], help='Confidence margin for CW Linf')
args = parser.parse_args([])

config.update(vars(args))

SEED = config['SEED']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


使用设备: cuda


In [2]:
if __name__ == "__main__":
    for dataset_name, (model_name, weights) in datasets.items():
        print(f"\n=== 开始实验: {dataset_name} 使用模型 {model_name} ===")

        model = getattr(models, model_name)(weights=weights).to(device)
        model.eval()

        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.ToTensor(),
        ])

        config['val_dir'] = f"./{dataset_name}"
        config['attack_name'] = f"cw_{model_name}"

        val_dir = config['val_dir']
        all_images = [f for f in os.listdir(val_dir) if f.endswith('.JPEG')]
        selected_images = random.sample(all_images, min(config['selected_count'], len(all_images)))

        output_dir = config['output_dir']
        attack_name = config['attack_name']
        attack_output_dir = os.path.join(output_dir, attack_name)
        os.makedirs(attack_output_dir, exist_ok=True)

        results = run_experiment(
            config=config,
            transform=transform,
            model=model,
            device=device,
            selected_images=selected_images,
            attack_output_dir=attack_output_dir,
            attack_cls=CWLinf,
        )

        print(f"=== 实验完成: {dataset_name} ===")



=== 开始实验: correct_1000_AlexNet 使用模型 alexnet ===


cw_alexnet: 100%|██████████| 500/500 [00:37<00:00, 13.38it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'cw_alexnet', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'kappa': 0.0, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_AlexNet'}
成功攻击: 497/500
攻击成功率: 99.40%
平均被修改像素点数量: 85886.48
平均被修改像素比例: 99.07%
平均扰动均值: 12272.44
平均 SSIM: 0.935023
平均 PSNR: 37.1213dB
SSIM >= 0.975: 13.68%
SSIM >= 0.98: 9.46%
SSIM >= 0.985: 4.02%
SSIM >= 0.99: 0.80%
SSIM >= 0.995: 0.00%
PSNR >= 39dB: 16.70%
PSNR >= 41dB: 3.22%
PSNR >= 43dB: 0.80%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_AlexNet ===

=== 开始实验: correct_1000_DenseNet121 使用模型 densenet121 ===


cw_densenet121: 100%|██████████| 500/500 [01:37<00:00,  5.14it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'cw_densenet121', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'kappa': 0.0, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_DenseNet121'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 86019.80
平均被修改像素比例: 99.85%
平均扰动均值: 9073.10
平均 SSIM: 0.962302
平均 PSNR: 39.8270dB
SSIM >= 0.975: 30.80%
SSIM >= 0.98: 20.60%
SSIM >= 0.985: 13.60%
SSIM >= 0.99: 4.60%
SSIM >= 0.995: 0.60%
PSNR >= 39dB: 72.80%
PSNR >= 41dB: 17.80%
PSNR >= 43dB: 2.40%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_DenseNet121 ===

=== 开始实验: correct_1000_GoogLeNet 使用模型 googlenet ===


...

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'cw_resnet34', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'kappa': 0.0, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_ResNet34'}
成功攻击: 498/500
攻击成功率: 99.60%
平均被修改像素点数量: 86471.46
平均被修改像素比例: 99.71%
平均扰动均值: 9289.05
平均 SSIM: 0.960050
平均 PSNR: 39.6150dB
SSIM >= 0.975: 28.51%
SSIM >= 0.98: 20.88%
SSIM >= 0.985: 13.05%
SSIM >= 0.99: 5.02%
SSIM >= 0.995: 0.80%
PSNR >= 39dB: 67.07%
PSNR >= 41dB: 17.47%
PSNR >= 43dB: 1.61%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_ResNet34 ===

=== 开始实验: correct_1000_VGG11 使用模型 vgg11 ===


cw_vgg11: 100%|██████████| 500/500 [01:16<00:00,  6.53it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'cw_vgg11', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'kappa': 0.0, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_VGG11'}
成功攻击: 499/500
攻击成功率: 99.80%
平均被修改像素点数量: 85912.76
平均被修改像素比例: 99.77%
平均扰动均值: 10061.11
平均 SSIM: 0.951124
平均 PSNR: 39.0021dB
SSIM >= 0.975: 24.85%
SSIM >= 0.98: 18.84%
SSIM >= 0.985: 11.22%
SSIM >= 0.99: 4.61%
SSIM >= 0.995: 0.40%
PSNR >= 39dB: 49.90%
PSNR >= 41dB: 15.83%
PSNR >= 43dB: 2.00%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_VGG11 ===

=== 开始实验: correct_1000_EfficientNet_b0 使用模型 efficientnet_b0 ===


cw_efficientnet_b0: 100%|██████████| 500/500 [01:51<00:00,  4.49it/s]

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'cw_efficientnet_b0', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'kappa': 0.0, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_EfficientNet_b0'}
成功攻击: 493/500
攻击成功率: 98.60%
平均被修改像素点数量: 86533.62
平均被修改像素比例: 99.76%
平均扰动均值: 13092.68
平均 SSIM: 0.923611
平均 PSNR: 36.4050dB
SSIM >= 0.975: 11.97%
SSIM >= 0.98: 6.29%
SSIM >= 0.985: 3.04%
SSIM >= 0.99: 1.01%
SSIM >= 0.995: 0.00%
PSNR >= 39dB: 12.78%
PSNR >= 41dB: 2.43%
PSNR >= 43dB: 0.20%
PSNR >= 45dB: 0.00%
PSNR >= 47dB: 0.00%
=== 实验完成: correct_1000_EfficientNet_b0 ===
